# Audio + Text + Dialogue Context (RoBERTa + WavLM)

context 추가

## 1. 환경 설정과 기본 설정

처음 점검할 때는 CONFIG 앞의 `SMOKE_TEST = True`로 실행합니다.
train/dev 각 4개로 forward, loss, backward와 평가를 확인합니다.
`False`로 되돌린 뒤 kernel을 재시작하고 위에서부터 실행합니다.


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


In [ ]:
import json
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoFeatureExtractor,
    AutoModel,
    AutoTokenizer,
    WavLMModel,
)


In [ ]:
# 모델, 입력, 학습에 사용할 기본 설정을 한곳에 모읍니다.
project_candidates = [
    Path.cwd(),
    Path("/workspace/emotion-ai"),
]
ROOT = next(
    (
        path
        for path in project_candidates
        if (path / "data/meld_metadata").is_dir()
    ),
    Path.cwd(),
)

SMOKE_TEST = False

CONFIG = {
    "experiment": "audio_text_dialogue_context",
    "context_size": 1,  # 최근 과거 발화 수: 1 / 3 / 5 중 선택
    "context_max_length": 128,  # 현재 발화 max_text_length와 별도 제한
    "context_policy": "same_split_same_dialogue_past_only",
    "context_order": "oldest_to_newest",
    "context_speaker_prefix": True,
    "condition": "clean",
    "text_model": "roberta-base",
    "audio_model": "microsoft/wavlm-base-plus",
    "text_revision": "main",
    "audio_revision": "main",
    "max_text_length": 128,
    "sample_rate": 16000,
    "dropout": 0.1,
    "batch_size": 2,
    "accumulation_steps": 4,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "betas": [0.9, 0.999],
    "eps": 1e-8,
    "epochs": 3,
    "seed": 42,
    "loss": "train_balanced_cross_entropy",
    "checkpoint_metric": "dev_macro_f1",
    "freeze_audio_feature_extractor": True,
    "apply_spec_augment": False,
    "gradient_checkpointing": True,
    "smoke_test": SMOKE_TEST,
}


In [ ]:
emotions = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise",
]

label2id = {
    emotion: index
    for index, emotion in enumerate(emotions)
}
id2label = {
    index: emotion
    for emotion, index in label2id.items()
}

CONFIG["label2id"] = label2id


In [ ]:
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
use_amp = device.type == "cuda"
amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16,
)

print("device:", device)
print("amp dtype:", amp_dtype if use_amp else None)


In [ ]:
run_suffix = "_smoke" if SMOKE_TEST else "_full"
run_id = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
) + run_suffix

run_dir = (
    ROOT
    / "experiments"
    / "audio_text_dialogue_context"
    / f"fusion_context_{CONFIG['context_size']}"
    / run_id
)
run_dir.mkdir(parents=True, exist_ok=False)


def save_json(filename, value):
    output_path = run_dir / filename
    temporary_path = output_path.with_suffix(
        output_path.suffix + ".tmp"
    )
    temporary_path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        ),
        encoding="utf-8",
    )
    temporary_path.replace(output_path)



In [ ]:
save_json("config.json", CONFIG)
print("결과 폴더:", run_dir)


## 2. 텍스트·오디오 데이터 정합성 검사


In [ ]:
KEYS = ["Dialogue_ID", "Utterance_ID"]
BAD_TEST_KEYS = {(38, 4), (220, 0)}

text_metadata_dir = ROOT / "data/meld_metadata"
audio_data_dir = ROOT / "data/meld_audio"
extracted_audio_dir = audio_data_dir / "extracted"


In [ ]:
def validate_pairing(text_dataframe, audio_dataframe):
    required_columns = [
        *KEYS,
        "Emotion",
        "Utterance",
        "Speaker",
    ]

    for source_name, dataframe in [
        ("text", text_dataframe),
        ("audio", audio_dataframe),
    ]:
        if not set(required_columns).issubset(dataframe.columns):
            raise ValueError(
                f"{source_name}: 필수 column 누락"
            )

        if dataframe[required_columns].isna().any().any():
            raise ValueError(
                f"{source_name}: 필수 값 누락"
            )

        for key in KEYS:
            valid_integer_key = pd.api.types.is_integer_dtype(
                dataframe[key]
            )
            if not valid_integer_key or (dataframe[key] < 0).any():
                raise ValueError(
                    f"{source_name}: {key}는 음수가 아닌 정수여야 합니다"
                )

        if dataframe.duplicated(KEYS).any():
            raise ValueError(
                f"{source_name}: 중복 utterance ID"
            )

        if not dataframe["Emotion"].isin(emotions).all():
            raise ValueError(
                f"{source_name}: 알 수 없는 emotion label"
            )

    paired = text_dataframe.merge(
        audio_dataframe[[*KEYS, "Emotion"]],
        on=KEYS,
        how="outer",
        suffixes=("", "_audio"),
        indicator=True,
        validate="one_to_one",
    )

    if not paired["_merge"].eq("both").all():
        raise ValueError("Text/Audio metadata ID 불일치")

    if not paired["Emotion"].eq(paired["Emotion_audio"]).all():
        raise ValueError("Text/Audio emotion label 불일치")

    # Text baseline과 같은 Utterance/Speaker 및 행 순서를 유지합니다.
    return text_dataframe.copy()


In [ ]:


def add_dialogue_context(dataframe, context_size):
    """Precompute past-only context once, never by scanning in __getitem__."""
    if context_size not in {1, 3, 5}:
        raise ValueError("context_size must be one of 1, 3, 5")
    if dataframe.duplicated(KEYS).any():
        raise ValueError("Context source contains duplicate utterance IDs")

    result = dataframe.copy()
    lookup = {}
    for dialogue_id, group in dataframe.groupby("Dialogue_ID", sort=False):
        ordered = group.sort_values("Utterance_ID", kind="stable")
        past = []
        for row in ordered.itertuples(index=False):
            previous = past[-context_size:]
            previous_ids = [item[0] for item in previous]
            if not all(value < row.Utterance_ID for value in previous_ids):
                raise AssertionError("Current/future utterance leaked into context")

            turn_texts = [item[1] for item in previous]
            lookup[(dialogue_id, row.Utterance_ID)] = {
                "context_previous_ids": json.dumps(previous_ids),
                "context_turn_texts": json.dumps(turn_texts, ensure_ascii=False),
                "context_text": "\n".join(turn_texts),
            }
            past.append(
                (row.Utterance_ID, f"{row.Speaker}: {row.Utterance}")
            )

    keys = list(dataframe[KEYS].itertuples(index=False, name=None))
    for column in ["context_previous_ids", "context_turn_texts", "context_text"]:
        result[column] = [lookup[key][column] for key in keys]
    return result


def validate_dialogue_context(dataframe, context_size, split):
    """Prove exact same-split/dialogue, strictly-past, recent-window usage."""
    if dataframe["split"].nunique() != 1 or dataframe["split"].iloc[0] != split:
        raise AssertionError("Context frame crossed an official split")

    for dialogue_id, group in dataframe.groupby("Dialogue_ID", sort=False):
        ordered = group.sort_values("Utterance_ID", kind="stable")
        all_ids = ordered["Utterance_ID"].tolist()
        for position, row in enumerate(ordered.itertuples(index=False)):
            actual = json.loads(row.context_previous_ids)
            expected = all_ids[max(0, position - context_size):position]
            if actual != expected:
                raise AssertionError(
                    f"{split} dialogue {dialogue_id} utterance "
                    f"{row.Utterance_ID}: context IDs {actual} != {expected}"
                )
            if not all(value < row.Utterance_ID for value in actual):
                raise AssertionError("Context is not strictly past-only")
            
def prepare_split(split):
    text_csv = text_metadata_dir / f"{split}_sent_emo.csv"
    audio_csv = audio_data_dir / f"{split}.csv"
    audio_dir = extracted_audio_dir / split

    if not audio_dir.is_dir():
        raise FileNotFoundError(
            f"오디오 압축 해제가 필요합니다: {audio_dir}"
        )

    text_dataframe = pd.read_csv(text_csv)
    audio_dataframe = pd.read_csv(audio_csv)
    dataframe = validate_pairing(
        text_dataframe,
        audio_dataframe,
    )
    dataframe["split"] = split
    dataframe = add_dialogue_context(
        dataframe, CONFIG["context_size"]
    )
    validate_dialogue_context(
        dataframe, CONFIG["context_size"], split
    )

    dataframe["audio_path"] = [
        str(audio_dir / f"dia{dialogue_id}_utt{utterance_id}.flac")
        for dialogue_id, utterance_id
        in dataframe[KEYS].itertuples(index=False, name=None)
    ]

    is_known_bad = pd.Series(
        [
            split == "test" and (dialogue_id, utterance_id) in BAD_TEST_KEYS
            for dialogue_id, utterance_id
            in dataframe[KEYS].itertuples(index=False, name=None)
        ],
        index=dataframe.index,
    )
    audio_exists = dataframe["audio_path"].map(
        lambda path: Path(path).is_file()
    )

    exclusion_reason = np.where(
        is_known_bad,
        "baseline_test_exclusion",
        np.where(~audio_exists, "missing_audio", ""),
    )

    excluded = dataframe.loc[
        exclusion_reason != "",
        [*KEYS, "split", "Emotion", "audio_path"],
    ].copy()
    excluded["reason"] = exclusion_reason[exclusion_reason != ""]

    if split == "test" and (~audio_exists & ~is_known_bad).any():
        raise FileNotFoundError(
            "공통 test에 알려지지 않은 누락 audio가 있습니다."
        )

    included = dataframe.loc[
        audio_exists & ~is_known_bad
    ].reset_index(drop=True)

    if included.empty:
        raise ValueError(f"{split}: 유효 sample 없음")

    if split == "test" and len(included) != 2608:
        raise ValueError(
            f"공통 test 수가 2608이 아닙니다: {len(included)}"
        )

    return included, excluded, text_csv, audio_csv


In [ ]:
frames = {}
audits = {}

for split in ["train", "dev", "test"]:
    frame, excluded, text_csv, audio_csv = prepare_split(split)

    for audio_path in tqdm(
        frame["audio_path"],
        desc=f"{split} audio headers",
    ):
        audio_info = sf.info(audio_path)
        valid_sample_rate = (
            audio_info.samplerate == CONFIG["sample_rate"]
        )
        valid_length = audio_info.frames >= 400

        if not valid_sample_rate or not valid_length:
            raise ValueError(
                f"16 kHz / WavLM 최소 길이 조건 오류: {audio_path}"
            )

    manifest_path = run_dir / f"{split}_manifest.csv"
    excluded_path = run_dir / f"{split}_excluded.csv"
    frame.to_csv(manifest_path, index=False)
    excluded.to_csv(excluded_path, index=False)

    frames[split] = frame
    audits[split] = {
        "original": len(frame) + len(excluded),
        "included": len(frame),
        "excluded": len(excluded),
        "text_csv": str(text_csv),
        "audio_csv": str(audio_csv),
    }

save_json("data_audit.json", audits)
print({split: len(frame) for split, frame in frames.items()})


# Inspect five targets under all candidate windows before any training.
inspection_source = pd.read_csv(text_metadata_dir / "train_sent_emo.csv")
inspection_source["split"] = "train"
inspection_group = next(
    group.sort_values("Utterance_ID", kind="stable")
    for _, group in inspection_source.groupby("Dialogue_ID", sort=False)
    if len(group) >= 6
)
inspection_rows = inspection_group.iloc[[0, 1, 2, 4, 5]][
    ["Dialogue_ID", "Utterance_ID", "Utterance", "Emotion"]
].copy()
for size in [1, 3, 5]:
    candidate = add_dialogue_context(inspection_source, size).set_index(KEYS)
    inspection_rows[f"context_{size}"] = [
        candidate.loc[(row.Dialogue_ID, row.Utterance_ID), "context_text"]
        for row in inspection_rows.itertuples(index=False)
    ]
inspection_records = inspection_rows.to_dict(orient="records")
save_json("context_inspection.json", inspection_records)
for row in inspection_records:
    print("CONTEXT INSPECTION", row["Dialogue_ID"], row["Utterance_ID"], row["Emotion"])
    print("TARGET:", row["Utterance"])
    for size in [1, 3, 5]:
        print(f"CONTEXT {size}:", row[f"context_{size}"] or "<EMPTY>")


## 3. Tokenizer와 Audio Feature Extractor


In [ ]:
# 사전학습된 tokenizer와 audio processor를 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["text_model"],
    revision=CONFIG["text_revision"],
)
audio_processor = AutoFeatureExtractor.from_pretrained(
    CONFIG["audio_model"],
    revision=CONFIG["audio_revision"],
)



## 4. Dataset과 Collator

In [ ]:
class MELDFusionDataset(Dataset):
    def __init__(self, dataframe):
        required_columns = [
            "split",
            "Dialogue_ID",
            "Utterance_ID",
            "Utterance",
            "Speaker",
            "Emotion",
            "audio_path",
            "context_text",
            "context_turn_texts",
            "context_previous_ids",
        ]

        missing_columns = set(required_columns) - set(dataframe.columns)
        if missing_columns:
            raise ValueError(
                f"Dataset 필수 column 누락: {sorted(missing_columns)}"
            )

        if dataframe[required_columns].isna().any().any():
            raise ValueError("Dataset 필수 값 누락")

        if dataframe["split"].nunique() != 1:
            raise ValueError("Dataset은 하나의 split만 포함해야 합니다")

        if dataframe.duplicated(KEYS).any():
            raise ValueError("Dataset에 중복 utterance ID가 있습니다")

        self.dataframe = dataframe.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        audio_path = Path(row["audio_path"])
        expected_filename = (
            f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.flac"
        )

        valid_audio_path = (
            audio_path.name == expected_filename
            and audio_path.parent.name == row["split"]
        )
        if not valid_audio_path:
            raise ValueError(
                f"Audio path/utterance ID 불일치: {audio_path}"
            )

        waveform, sample_rate = sf.read(
            audio_path,
            dtype="float32",
        )
        if waveform.ndim == 2:
            waveform = waveform.mean(axis=1)

        valid_waveform = (
            sample_rate == CONFIG["sample_rate"]
            and len(waveform) >= 400
            and np.isfinite(waveform).all()
        )
        if not valid_waveform:
            raise ValueError(f"유효하지 않은 waveform: {audio_path}")

        encoded_text = tokenizer(
            str(row["Utterance"]),
            truncation=True,
            padding="max_length",
            max_length=CONFIG["max_text_length"],
            return_tensors="pt",
        )

        context_turn_texts = json.loads(row["context_turn_texts"])
        # Reuse RoBERTa's real separator token; do not add a new token.
        context_model_text = f" {tokenizer.sep_token} ".join(context_turn_texts)
        encoded_context = tokenizer(
            context_model_text,
            truncation=True,
            padding="max_length",
            max_length=CONFIG["context_max_length"],
            return_tensors="pt",
        )

        return {
            "input_ids": encoded_text["input_ids"].squeeze(0),
            "text_attention_mask": encoded_text["attention_mask"].squeeze(0),
            "context_input_ids": encoded_context["input_ids"].squeeze(0),
            "context_attention_mask": encoded_context["attention_mask"].squeeze(0),
            "waveform": waveform,
            "label": label2id[row["Emotion"]],
            "split": str(row["split"]),
            "Dialogue_ID": int(row["Dialogue_ID"]),
            "Utterance_ID": int(row["Utterance_ID"]),
            "Utterance": str(row["Utterance"]),
            "Speaker": str(row["Speaker"]),
            "context_text": str(row["context_text"]),
            "context_previous_ids": str(row["context_previous_ids"]),
            "audio_path": str(audio_path),
        }


In [ ]:
class FusionCollator:
    def __init__(self, audio_processor, sample_rate=16000):
        self.audio_processor = audio_processor
        self.sample_rate = sample_rate

    def __call__(self, batch):
        audio_inputs = self.audio_processor(
            [sample["waveform"] for sample in batch],
            sampling_rate=self.sample_rate,
            padding=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "input_ids": torch.stack(
                [sample["input_ids"] for sample in batch]
            ),
            "text_attention_mask": torch.stack(
                [sample["text_attention_mask"] for sample in batch]
            ),
            "context_input_ids": torch.stack(
                [sample["context_input_ids"] for sample in batch]
            ),
            "context_attention_mask": torch.stack(
                [sample["context_attention_mask"] for sample in batch]
            ),
            "input_values": audio_inputs["input_values"],
            "audio_attention_mask": audio_inputs["attention_mask"],
            "labels": torch.tensor(
                [sample["label"] for sample in batch],
                dtype=torch.long,
            ),
        }


In [ ]:
used_frames = {
    split: (
        frame.head(4).reset_index(drop=True)
        if SMOKE_TEST and split != "test"
        else frame
    )
    for split, frame in frames.items()
}

datasets = {
    split: MELDFusionDataset(frame)
    for split, frame in used_frames.items()
    if not SMOKE_TEST or split != "test"
}

print({split: len(dataset) for split, dataset in datasets.items()})


## 5. DataLoader

In [ ]:
fusion_collator = FusionCollator(
    audio_processor=audio_processor,
    sample_rate=CONFIG["sample_rate"],
)

loaders = {}
for split, dataset in datasets.items():
    used_frames[split].to_csv(
        run_dir / f"{split}_used_manifest.csv",
        index=False,
    )
    loaders[split] = DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=split == "train",
        collate_fn=fusion_collator,
        num_workers=0,
        generator=torch.Generator().manual_seed(CONFIG["seed"]),
    )

save_json(
    "used_sample_counts.json",
    {
        split: len(loader.dataset)
        for split, loader in loaders.items()
    },
)


## 6. Modeling

현재 text, audio, context 순서로 concatenate합니다.
기존 실행 shape는 [B, 768] × 3 → [B, 2304] → [B, 7]


In [ ]:
# RoBERTa와 WavLM 표현을 concatenate하는 감정 분류 모델을 정의합니다.
class AudioTextFusionClassifier(nn.Module):
    def __init__(
        self,
        text_model_name="roberta-base",
        audio_model_name="microsoft/wavlm-base-plus",
        num_classes=7,
        dropout=0.1,
    ):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(
            text_model_name,
            revision=CONFIG["text_revision"],
        )
        self.audio_encoder = WavLMModel.from_pretrained(
            audio_model_name,
            revision=CONFIG["audio_revision"],
        )

        # Current/context share this single text encoder instance.
        text_width = self.text_encoder.config.hidden_size
        audio_width = self.audio_encoder.config.hidden_size
        fusion_width = text_width + audio_width + text_width
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            fusion_width,
            num_classes,
        )

    def forward(
        self,
        input_ids,
        text_attention_mask,
        context_input_ids,
        context_attention_mask,
        input_values,
        audio_attention_mask,
        return_shapes=False,
    ):
        # RoBERTa의 첫 token(<s>) 표현을 문장 표현으로 사용합니다.
        text_hidden_states = self.text_encoder(
            input_ids=input_ids,
            attention_mask=text_attention_mask,
        ).last_hidden_state
        text_embedding = text_hidden_states[:, 0, :]  # [B, text_hidden]

        # Same RoBERTa instance, called again for context.
        context_hidden_states = self.text_encoder(
            input_ids=context_input_ids,
            attention_mask=context_attention_mask,
        ).last_hidden_state
        context_embedding = context_hidden_states[:, 0, :]  # [B, text_hidden]

        # WavLM 출력의 padding 위치를 제외하고 시간축 평균을 계산합니다.
        audio_hidden_states = self.audio_encoder(
            input_values=input_values,
            attention_mask=audio_attention_mask,
        ).last_hidden_state
        feature_mask = self.audio_encoder._get_feature_vector_attention_mask(
            audio_hidden_states.shape[1],
            audio_attention_mask,
        )
        feature_mask = feature_mask.unsqueeze(-1).to(
            audio_hidden_states.dtype
        )
        audio_embedding = (
            (audio_hidden_states * feature_mask).sum(dim=1)
            / feature_mask.sum(dim=1).clamp(min=1)
        )

        # audio_embedding: [B, audio_hidden]
        fusion_embedding = torch.cat(
            [text_embedding, audio_embedding, context_embedding],
            dim=-1,
        )  # [B, text_hidden + audio_hidden + text_hidden]
        # logits: [B, num_classes], MELD에서는 [B, 7]
        logits = self.classifier(
            self.dropout(fusion_embedding)
        )

        if return_shapes:
            shapes = {
                "text_embedding": list(text_embedding.shape),
                "audio_embedding": list(audio_embedding.shape),
                "context_embedding": list(context_embedding.shape),
                "fusion_embedding": list(fusion_embedding.shape),
                "logits": list(logits.shape),
            }
            return logits, shapes

        return logits


In [ ]:
# 본 학습에 사용할 Fusion 모델을 생성합니다.
model = AudioTextFusionClassifier(
    text_model_name=CONFIG["text_model"],
    audio_model_name=CONFIG["audio_model"],
    num_classes=len(emotions),
    dropout=CONFIG["dropout"],
)


In [ ]:
# CONFIG에 따라 WavLM 앞단을 고정하고 메모리 절약 설정을 적용합니다.
if CONFIG["freeze_audio_feature_extractor"]:
    for parameter in model.audio_encoder.feature_extractor.parameters():
        parameter.requires_grad = False

model.audio_encoder.config.apply_spec_augment = CONFIG[
    "apply_spec_augment"
]

if CONFIG["gradient_checkpointing"]:
    model.text_encoder.gradient_checkpointing_enable()
    model.audio_encoder.gradient_checkpointing_enable()

model = model.to(device)


In [ ]:
parameter_counts = {
    "total": sum(
        parameter.numel()
        for parameter in model.parameters()
    ),
    "trainable": sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
}
save_json("parameter_counts.json", parameter_counts)

print("total parameters:", f"{parameter_counts['total']:,}")
print("trainable parameters:", f"{parameter_counts['trainable']:,}")


## 7. Class Weights, Loss, Optimizer

In [ ]:
train_label_ids = frames["train"]["Emotion"].map(
    label2id
).to_numpy()

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(emotions)),
    y=train_label_ids,
)
class_weight_table = dict(
    zip(emotions, class_weights.tolist())
)

save_json("class_weights.json", class_weight_table)
pd.Series(class_weight_table, name="weight")


In [ ]:
class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device,
)
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ],
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    betas=tuple(CONFIG["betas"]),
    eps=CONFIG["eps"],
)


## 8. Forward 연결 확인

In [ ]:
# 첫 batch를 학습 없이 forward하여 각 표현의 크기를 확인합니다.
def check_forward_shapes(model, loader, device):
    batch = next(iter(loader))
    input_names = [
        "input_ids",
        "text_attention_mask",
        "context_input_ids",
        "context_attention_mask",
        "input_values",
        "audio_attention_mask",
    ]
    inputs = {
        name: batch[name].to(device)
        for name in input_names
    }

    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            logits, shapes = model(
                **inputs,
                return_shapes=True,
            )
    finally:
        model.train(was_training)

    batch_size = inputs["input_ids"].shape[0]
    text_width = model.text_encoder.config.hidden_size
    audio_width = model.audio_encoder.config.hidden_size

    assert shapes["text_embedding"] == [batch_size, text_width]
    assert shapes["audio_embedding"] == [batch_size, audio_width]
    assert shapes["context_embedding"] == [batch_size, text_width]
    assert shapes["fusion_embedding"] == [
        batch_size,
        text_width + audio_width + text_width,
    ]
    assert shapes["logits"] == [batch_size, len(emotions)]
    assert torch.isfinite(logits).all()

    return {
        "status": "passed",
        "shapes": shapes,
        "finite_logits": True,
        "training_steps": 0,
    }


In [ ]:
forward_check = check_forward_shapes(
    model,
    loaders["train"],
    device,
)
save_json("forward_smoke.json", forward_check)
print(forward_check["shapes"])


context_audit = {}
for split, frame in frames.items():
    official = pd.read_csv(
        text_metadata_dir / f"{split}_sent_emo.csv"
    )
    official["split"] = split
    expected_context = add_dialogue_context(
        official, CONFIG["context_size"]
    )
    validate_dialogue_context(
        expected_context, CONFIG["context_size"], split
    )
    expected_ids = expected_context.set_index(KEYS)[
        "context_previous_ids"
    ]
    for row in frame.itertuples(index=False):
        expected = expected_ids.loc[
            (row.Dialogue_ID, row.Utterance_ID)
        ]
        if row.context_previous_ids != expected:
            raise AssertionError(
                f"{split} target context differs from official text context"
            )
        if not all(
            value < row.Utterance_ID
            for value in json.loads(row.context_previous_ids)
        ):
            raise AssertionError("Target context is not strictly past-only")
    counts = frame["context_previous_ids"].map(lambda value: len(json.loads(value)))
    context_audit[split] = {
        "sample_count": len(frame),
        "empty_context_count": int((counts == 0).sum()),
        "short_context_count": int((counts < CONFIG["context_size"]).sum()),
        "full_context_count": int((counts == CONFIG["context_size"]).sum()),
        "maximum_previous_turns": int(counts.max()),
        "past_only_same_dialogue_same_split": True,
    }
save_json("context_audit.json", context_audit)
print("context audit:", context_audit)


## 9. 학습·평가 함수

In [ ]:
def move_batch_to_device(batch, device):
    return {
        name: tensor.to(device)
        for name, tensor in batch.items()
    }


In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    accumulation_steps=4,
):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    weighted_loss_sum = 0.0
    class_weight_sum = 0.0
    number_of_batches = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="Training")):
        batch = move_batch_to_device(batch, device)
        labels = batch.pop("labels")

        # 마지막 accumulation 묶음이 짧아도 gradient 크기를 유지합니다.
        group_start = (step // accumulation_steps) * accumulation_steps
        group_size = min(
            accumulation_steps,
            number_of_batches - group_start,
        )

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = model(**batch)
            loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise FloatingPointError("non-finite training loss")

        scaler.scale(loss / group_size).backward()

        update_now = (
            (step + 1) % accumulation_steps == 0
            or step + 1 == number_of_batches
        )
        if update_now:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_weight = criterion.weight[labels].sum().item()
        weighted_loss_sum += loss.item() * batch_weight
        class_weight_sum += batch_weight

    return weighted_loss_sum / class_weight_sum


In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    weighted_loss_sum = 0.0
    class_weight_sum = 0.0
    true_labels = []
    predicted_labels = []

    for batch in tqdm(loader, desc="Evaluation"):
        batch = move_batch_to_device(batch, device)
        labels = batch.pop("labels")
        logits = model(**batch)

        if not torch.isfinite(logits).all():
            raise FloatingPointError("non-finite evaluation logits")

        loss = criterion(logits, labels)
        batch_weight = criterion.weight[labels].sum().item()
        weighted_loss_sum += loss.item() * batch_weight
        class_weight_sum += batch_weight

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(
            logits.argmax(dim=1).cpu().tolist()
        )

    macro_f1 = f1_score(
        true_labels,
        predicted_labels,
        labels=list(range(len(emotions))),
        average="macro",
        zero_division=0,
    )

    return (
        weighted_loss_sum / class_weight_sum,
        float(macro_f1),
        true_labels,
        predicted_labels,
    )


## 10. Checkpoint와 Main Training


In [ ]:
checkpoint_path = run_dir / "best.pt"
history = []
best_dev_f1 = -1.0

save_json(
    "status.json",
    {"status": "training", "test_macro_f1": None},
)


In [ ]:
number_of_epochs = 1 if SMOKE_TEST else CONFIG["epochs"]

for epoch in range(1, number_of_epochs + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=loaders["train"],
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        accumulation_steps=CONFIG["accumulation_steps"],
    )
    dev_loss, dev_f1, _, _ = evaluate(
        model=model,
        loader=loaders["dev"],
        criterion=criterion,
        device=device,
    )

    epoch_result = {
        "epoch": epoch,
        "train_loss": train_loss,
        "dev_loss": dev_loss,
        "dev_macro_f1": dev_f1,
        "learning_rate": optimizer.param_groups[0]["lr"],
    }
    history.append(epoch_result)

    save_json("history.json", history)
    pd.DataFrame(history).to_csv(
        run_dir / "history.csv",
        index=False,
    )

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        torch.save(
            {
                "epoch": epoch,
                "dev_macro_f1": dev_f1,
                "config": CONFIG,
                "model_state_dict": model.state_dict(),
            },
            checkpoint_path,
        )

    print(epoch_result)

save_json(
    "status.json",
    {
        "status": "trained",
        "best_dev_macro_f1": best_dev_f1,
        "test_macro_f1": None,
    },
)


## 11. Best Checkpoint 평가와 예측 저장

In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)
model.load_state_dict(checkpoint["model_state_dict"])
best_epoch = checkpoint["epoch"]
del checkpoint

evaluation_split = "dev" if SMOKE_TEST else "test"
evaluation_loss, macro_f1, true_labels, predicted_labels = evaluate(
    model=model,
    loader=loaders[evaluation_split],
    criterion=criterion,
    device=device,
)


In [ ]:
report = classification_report(
    true_labels,
    predicted_labels,
    labels=list(range(len(emotions))),
    target_names=emotions,
    output_dict=True,
    zero_division=0,
)
confusion = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=list(range(len(emotions))),
)

metrics = {
    "split": evaluation_split,
    "loss": evaluation_loss,
    "accuracy": float(
        np.mean(
            np.array(true_labels)
            == np.array(predicted_labels)
        )
    ),
    "macro_f1": macro_f1,
    "best_epoch": best_epoch,
    "sample_count": len(true_labels),
    "classification_report": report,
}

save_json(f"{evaluation_split}_metrics.json", metrics)
pd.DataFrame(report).transpose().to_csv(
    run_dir / f"{evaluation_split}_classification_report.csv"
)
pd.DataFrame(
    confusion,
    index=emotions,
    columns=emotions,
).to_csv(
    run_dir / f"{evaluation_split}_confusion_matrix.csv"
)


save_json(f"{evaluation_split}_y_true.json", true_labels)
save_json(f"{evaluation_split}_y_pred.json", predicted_labels)


In [ ]:
predictions = used_frames[evaluation_split][
    [
        "split", *KEYS, "Utterance", "Speaker",
        "context_text", "context_previous_ids",
        "Emotion", "audio_path",
    ]
].copy()

expected_labels = predictions["Emotion"].map(label2id).tolist()
assert expected_labels == true_labels, (
    "예측 결과와 manifest의 sample 순서가 다릅니다"
)

predictions["true_id"] = true_labels
predictions["fusion_pred_id"] = predicted_labels
predictions["fusion_prediction"] = [
    id2label[prediction_id]
    for prediction_id in predicted_labels
]
predictions.to_csv(
    run_dir / f"{evaluation_split}_predictions.csv",
    index=False,
)


In [ ]:
final_status = {
    "status": "smoke_complete" if SMOKE_TEST else "complete",
    "best_epoch": best_epoch,
    "best_dev_macro_f1": best_dev_f1,
    "test_macro_f1": None if SMOKE_TEST else macro_f1,
}
save_json("status.json", final_status)

print("best epoch:", best_epoch)
print("best dev Macro F1:", f"{best_dev_f1:.6f}")
print(f"{evaluation_split} Macro F1:", f"{macro_f1:.6f}")
print(
    classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(len(emotions))),
        target_names=emotions,
        digits=4,
        zero_division=0,
    )
)
print("artifacts:", run_dir)
